# 导出并回测 GP 公式

这个 Notebook 支持两件事：

1. 从已冻结的 `frozen.json` 或训练候选 `training_candidates.json` 中选择一个完整 expression hash，导出为普通因子 `.py` 文件。
2. 输入导出公式、H5 和任意回测时间段的**完整路径**，通过 `FactorManager` 回测并生成 HTML 报告。

注意：如果回测区间与 2026 测试期重叠，必须显式设置 `ALLOW_HOLDOUT_REPEAT = True`。这属于重复访问 holdout，不能用于后续选因子。

## 1. 初始化仓库环境

In [1]:
from pathlib import Path
import importlib
import json
import os
import sys

ipython = get_ipython()
if ipython is not None:
    if 'IPython.extensions.autoreload' not in ipython.extension_manager.loaded:
        ipython.run_line_magic('load_ext', 'autoreload')
    ipython.run_line_magic('autoreload', '2')

def find_repo_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    injected = os.environ.get('FACTOR_COMMON_REPO_ROOT')
    if injected:
        candidates.insert(0, Path(injected).expanduser().resolve())
    candidates.append(Path('/Users/dmiwu/work/PythonProject/cryptoFactorAnalyze'))
    for candidate in candidates:
        if (candidate / 'factor_common').is_dir() and (candidate / 'Genetic_Algorithm').is_dir():
            return candidate
    raise RuntimeError('无法定位 cryptoFactorAnalyze 仓库根目录')

ROOT = find_repo_root(Path.cwd())
repo_path = str(ROOT)
sys.path[:] = [item for item in sys.path if item != repo_path]
sys.path.insert(0, repo_path)
for module_name in tuple(sys.modules):
    if module_name in {'factor_common', 'Genetic_Algorithm'} or module_name.startswith(('factor_common.', 'Genetic_Algorithm.')):
        del sys.modules[module_name]

print('repo root:', ROOT)

repo root: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze


## 2. 输入完整路径与回测参数

把下面所有路径替换为真实的**绝对路径**。若只想回测已有 `.py` 公式，可先跳过导出单元。

In [2]:
# 导出某个冻结候选时使用。
CANDIDATE_MANIFEST_PATH = Path('/Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/Genetic_Algorithm/runs/daily_continuous_multiseed_20260916T033639Z/pooled/training_candidates.json')
EXPRESSION_ID = '77282e814dac81a8bc35a65b4b7e8c3b029d6f0426e836a3911b07f4a230cf94'
EXPORT_DIR = Path('/Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/factor_analyse/factor_mining/GP_Factor')

In [3]:
# 回测已有或刚导出的公式时使用。
FACTOR_PATH = Path('/Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/Genetic_Algorithm/runs/daily_continuous_multiseed_20260916T033639Z/pooled/exported_factors/GP_77282e814dac81a8.py')
H5_PATH = Path('/Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/data/crypto_quant.h5')
START_DATE = '2024-01-01'
END_DATE = '2026-08-31'
REBALANCE_DAYS = 1
N_GROUPS = 5
FEE_RATE = 0.0005
SLIPPAGE = 0.001
INCLUDE_FUNDING = True

# 任何与 2026-01-01 至 2026-09-01 相交的回测都必须明确确认。
ALLOW_HOLDOUT_REPEAT = True

def require_absolute(path: Path, label: str, *, exists: bool = True) -> Path:
    path = Path(path).expanduser()
    if not path.is_absolute():
        raise ValueError(f'{label} 必须是完整绝对路径: {path}')
    if exists:
        return path.resolve(strict=True)
    return path.resolve()

if REBALANCE_DAYS < 1:
    raise ValueError('REBALANCE_DAYS 必须至少为 1')
if N_GROUPS < 2:
    raise ValueError('N_GROUPS 必须至少为 2')

## 3. 列出冻结公式并导出指定公式

运行本单元格会校验候选 JSON 的 SHA-256，并显示可导出的候选。随后仅导出 `EXPRESSION_ID` 对应的公式；已有完全相同的导出文件会安全复用。训练候选可供研究回放，但只有冻结候选才可被称为独立 holdout 评估候选。

In [4]:
import pandas as pd
from Genetic_Algorithm.artifacts import read_verified_manifest
from Genetic_Algorithm.export import export_factor

manifest_path = require_absolute(CANDIDATE_MANIFEST_PATH, 'CANDIDATE_MANIFEST_PATH')
export_dir = require_absolute(EXPORT_DIR, 'EXPORT_DIR', exists=False)
manifest = read_verified_manifest(manifest_path)
candidates = manifest.get('candidates', [])
if not candidates:
    raise ValueError('候选清单没有公式；请使用包含 candidate 的 frozen.json 或 training_candidates.json')

display(pd.DataFrame([{
    'expression_id': item['expression_id'],
    'direction': item.get('direction', item.get('training_direction')),
    'ast': json.dumps(item['ast'], ensure_ascii=False),
} for item in candidates]))

candidate = next((item for item in candidates if item['expression_id'] == EXPRESSION_ID), None)
if candidate is None:
    raise KeyError('EXPRESSION_ID 不在候选清单中；请从上表复制完整 hash')

direction = candidate.get('direction', candidate.get('training_direction'))
if direction not in (-1, 1):
    raise ValueError('候选缺少有效的冻结/训练方向')
exported = export_factor(candidate, export_dir, direction=direction)
FACTOR_PATH = exported.path
print('formula:', candidate['expression_id'])
print('direction:', direction)
print('exported factor:', FACTOR_PATH)
print('export manifest:', exported.manifest_path)

,expression_id,direction,ast
0,4c00b7f073adb9414b90b96b57236b22a1c29509108128...,1,"{""children"": [{""children"": [{""children"": [{""ch..."
1,768b444e49c87ffcc56a1206439529e9a6521dee066388...,-1,"{""children"": [{""children"": [{""children"": [{""ch..."
2,327dc3eb17d3b874a50109240d9128311a752338d2a7fb...,1,"{""children"": [{""children"": [{""children"": [{""ch..."
3,d37aee657818a746bc3f1e93d434beea6dd8e7114f8b6e...,1,"{""children"": [{""children"": [{""children"": [], ""..."
4,f92c8150360cc58a7d9c911153761a38c3394e19671075...,-1,"{""children"": [{""children"": [{""children"": [], ""..."
5,56797009f8dcb527f5172bf573a6af3c8ef8a50adc93c9...,1,"{""children"": [{""children"": [{""children"": [{""ch..."
6,a81fccda502f3e569b56697da43c54479400795e8e1665...,1,"{""children"": [{""children"": [{""children"": [{""ch..."
7,f266b6b9adad55a51360292d054e436dc6c70a23541b83...,1,"{""children"": [{""children"": [{""children"": [{""ch..."
8,f54c546d39a692f50ef2d102e3fa8d79762f04483bd6d7...,1,"{""children"": [{""children"": [{""children"": [{""ch..."
9,f83dbbf325f609e9e7c3b8a8159ccd2e7b73c3007b61a0...,1,"{""children"": [{""children"": [{""children"": [{""ch..."


formula: 77282e814dac81a8bc35a65b4b7e8c3b029d6f0426e836a3911b07f4a230cf94
direction: -1
exported factor: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/factor_analyse/factor_mining/GP_Factor/GP_77282e814dac81a8.py
export manifest: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/factor_analyse/factor_mining/GP_Factor/GP_77282e814dac81a8.export_manifest.json


## 4. 用任意日期区间回测导出公式

In [5]:
from factor_common import FactorManager

factor_path = require_absolute(FACTOR_PATH, 'FACTOR_PATH')
h5_path = require_absolute(H5_PATH, 'H5_PATH')
start = pd.Timestamp(START_DATE).normalize()
end = pd.Timestamp(END_DATE).normalize()
if start > end:
    raise ValueError('START_DATE 不能晚于 END_DATE')

holdout_start = pd.Timestamp('2026-01-01')
holdout_end = pd.Timestamp('2026-09-01')
if start <= holdout_end and end >= holdout_start and not ALLOW_HOLDOUT_REPEAT:
    raise PermissionError(
        '该区间会读取 2026 holdout；如确认这是重复评估，请显式设置 ALLOW_HOLDOUT_REPEAT = True'
    )

manager = FactorManager(
    h5_path=h5_path,
    base_dir=ROOT / 'data' / 'factor_results',
    reports_dir=ROOT / 'reports',
    persist_evaluations=True,
)
_, market_end = manager.dp.get_time_range()
safe_signal_end = market_end - pd.Timedelta(days=1 + REBALANCE_DAYS)
if end > safe_signal_end:
    raise ValueError(
        f'END_DATE={end.date()} 需要平仓价格，但可完整结算的最后信号日为 {safe_signal_end.date()}'
    )

result = manager.evaluate(
    factor_path,
    params={
        'start': start.date().isoformat(),
        'end': end.date().isoformat(),
        'rebalance_days': REBALANCE_DAYS,
        'n_groups': N_GROUPS,
        'fee_rate': FEE_RATE,
        'slippage': SLIPPAGE,
        'include_funding': INCLUDE_FUNDING,
    },
    plot=False,
)

report_path = ROOT / 'reports' / f'{factor_path.stem}_{start:%Y%m%d}_{end:%Y%m%d}_{result["run_id"]}.html'
report_info = manager.plot_result(result, output_path=report_path)
print('status:', result['status'])
print('run_id:', result['run_id'])
print('evaluation_id:', result['evaluation_id'])
print('report:', report_info['output_path'])

status: complete
run_id: 0ec9c4096774ca44
evaluation_id: 3d5e45199e92cac9
report: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/reports/GP_77282e814dac81a8_20240101_20260831_0ec9c4096774ca44.html


## 5. 查看因子、IC、扣费后收益与因果性诊断

In [6]:
display(result['factor_value'].tail())

full_ic = result['factor_performance']['samples']['full']['ic']
all_costs = result['factor_performance']['scenarios']['all_costs']['full']
print('IC:', {key: full_ic.get(key) for key in ('ic_mean', 'rank_ic_mean', 'icir', 't_stat', 'n_dates')})
print('all_costs:', {key: all_costs.get(key) for key in ('total_return', 'annual_return', 'sharpe', 'max_drawdown', 'turnover', 'n_periods')})

validation = result['diagnostics']['validation']
print('static scan:', validation['static_scan']['status'])
print('cutoff replay:', validation['cutoff']['status'])
print('cutoff max_abs_diff:', [item['max_abs_diff'] for item in validation['cutoff'].get('cutoffs', [])])
print('funding coverage:', result['diagnostics']['coverage']['funding']['status_counts'])

,1INCHUSDT,2ZUSDT,AAVEUSDT,ADAUSDT,AGIXUSDT,ALGOUSDT,APEUSDT,APTUSDT,ARBUSDT,ARUSDT,...,XLMUSDT,XMRUSDT,XPLUSDT,XRPUSDT,XTZUSDT,ZECUSDT,ZKJUSDT,ZKUSDT,ZROUSDT,币安人生USDT
date,,,,,,,,,,,,,,,,,,,,,
2026-08-27,NaN,NaN,0.817919,0.758875,NaN,0.81237,NaN,0.690697,0.800056,NaN,...,0.882306,0.776677,NaN,0.764901,NaN,0.733757,NaN,NaN,NaN,0.932173
2026-08-28,NaN,NaN,0.817919,0.758875,NaN,0.81265,NaN,0.689803,0.800056,NaN,...,0.880493,0.763208,NaN,0.760768,NaN,0.730466,NaN,NaN,NaN,0.932173
2026-08-29,NaN,NaN,0.817919,0.758875,NaN,0.81293,NaN,0.688909,0.800056,NaN,...,0.878680,0.749740,NaN,0.756635,NaN,0.725959,NaN,NaN,NaN,0.932173
2026-08-30,NaN,NaN,0.817919,0.758875,NaN,0.81321,NaN,0.688016,0.800056,NaN,...,0.876867,0.738760,NaN,0.752591,NaN,0.721452,NaN,NaN,NaN,0.932173
2026-08-31,NaN,NaN,0.817919,0.758875,NaN,0.81349,NaN,0.687122,0.800177,NaN,...,0.875605,0.730269,NaN,0.748637,NaN,0.716945,NaN,NaN,NaN,0.932173


IC: {'ic_mean': -0.01836875387052594, 'rank_ic_mean': -0.01357080210618124, 'icir': -0.1025186278149193, 't_stat': -3.1978582183595146, 'n_dates': 973}
all_costs: {'total_return': 0.7611562524309712, 'annual_return': 0.2362637032930841, 'sharpe': 1.309961924014964, 'max_drawdown': 0.19648218570917433, 'turnover': 0.0967246575258499, 'n_periods': 974}
static scan: clean
cutoff replay: verified
cutoff max_abs_diff: [0.0, 0.0]
funding coverage: {'complete': 19365}
